In [ ]:
import numpy as np
import pandas as pd
from scipy.stats import pearsonr, spearmanr

import json5


from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score,mean_absolute_error, mean_squared_error, r2_score, classification_report
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.neural_network import MLPClassifier


from pykeen.pipeline import pipeline
from pykeen.datasets import get_dataset
from pykeen import predict
from pykeen.triples import TriplesFactory


import torch
from typing import List
import torch.nn.functional as F

from sklearn.model_selection import GridSearchCV

import joblib

# Prepare Data
We have 2 files:
- pairs: contains two columns one for each article id and at least one column containing the labels of relevance (can be a continuous score or a binary variable 0 or 1)
- news: contains the metadata of all news from the dataset, one column with the ids and column with the news metadata (could be text, recognized entities)

Pairs Example:

| art1 | art2 | continuous_score | binary_score |
|----------|----------|------------------|--------------|
| 101      | 205      | 0.85             | 1            |
| 102      | 207      | 0.30             | 0            |
| 103      | 208      | 0.65             | 1            |
| 104      | 209      | 0.10             | 0            |

News Example:
| article_id | entities          |
|-------------|-------------------|
| 101         | [{'surfaceForm': 'Apple','dbpedia_URI': 'http://dbpedia.org/resource/Apple_Inc.','wikidataId': 'Q312'},{'surfaceForm': 'iPad','dbpedia_URI': 'http://dbpedia.org/resource/IPad','wikidataId': 'Q2796'}]|
| 205         | [{'surfaceForm': 'Google','dbpedia_URI': 'http://dbpedia.org/resource/Google','wikidataId': 'Q95'},{'surfaceForm': 'Android','dbpedia_URI':'http://dbpedia.org/resource/Android_(operating_system)','wikidataId': 'Q26635'}]|

In the rest of this Notebook we will be using the CNRec dataset (cite).

In [ ]:
# DATA PREP
path_2_pairs = "DATASETS/CNRec/CNRec_All_Data_grouped.csv"
path_2_news = "DATASETS/CNRec/NER-corregido-wikidata-v2.tsv"

pairs=pd.read_csv(path_2_pairs, index_col=[0]).dropna().rename(columns={"rating":"simRating"})
news=pd.read_csv(path_2_news, sep="\t", index_col=[0])

# parse entities
news["Title entities"]=news["Title entities"].apply(lambda x: json5.loads(x))
news["text entities"]=news["text entities"].apply(lambda x: json5.loads(x))


train_pairs,test_pairs=train_test_split(pairs)

# Recommenders

In [ ]:
# uncomment in case you want to use your own pretrained embeddings
global_pretrained_embeddings = None

# with open('PRETRAINED_EMBEDDINGS/wikidata5m_embeddings.pkl', 'rb') as f:
#     print("LOADING EMBEDDINGS DICTIONARY")
#     global_pretrained_embeddings = pickle.load(f)
#     #CAMBIAR A TENSORES EN VEZ NUMPY ARRAYS
#     for k in global_pretrained_embeddings:
#         global_pretrained_embeddings[k]=torch.tensor(global_pretrained_embeddings[k])
#     print("EMBEDDINGS LOADED SUCCESFULLY")

In [ ]:
class Recommender():
    def __init__(self,name) -> None:
        self.name=name
        self.metrics=None

    def train(self, train_data, labels, task="classification"):
        '''Train the model parameters.'''
        raise NotImplementedError("Training method belongs to an abstract class.")
    
    def predict(self,test_data,task="classification"):
        '''Use the trained model to get predictions. It should return a value between 0 and 1 indicating how good is the recommendation.'''
        raise NotImplementedError("Predict method belongs to an abstract class.")
    
class KGERecommender(Recommender):
    '''Recommender that uses entity and relation embeddings as inputs.'''
    def __init__(self, name, triples_factory, use_pretrained_embeddings=False, entity_col="Title entities") -> None:
        super().__init__(name)
        self.kge_model=None
        self.triples_factory=triples_factory
        self.entity_col=entity_col
        self.use_pretrained_embeddings=use_pretrained_embeddings
        self.pretrained_embeddings=None

    def calculate_overlap(self,row,centers,radios):

        center1=centers[int(row["art1"])]
        center2=centers[int(row["art2"])]

        radius1=radios[int(row["art1"])]
        radius2=radios[int(row["art2"])]

        #print("CENTERS:",center_1,center_2)

        # if there is no data in any of the clusters
        if center1==None or center2==None:
            return 0
        
        # if one cluster is a single point and the other is not
        if radius1 == 0 and radius2 > 0:
            distance = np.linalg.norm(np.array(center2) - np.array(center1))
            if distance <= radius2:
                return 1.0
            else:
                return 0.0
        elif radius2 == 0 and radius1 > 0:
            distance = np.linalg.norm(np.array(center1) - np.array(center2))
            if distance <= radius1:
                return 1.0
            else:
                return 0.0
        
        # if both clusters are single points
        if (radius1 == radius2 == 0):
            if center1!=center2:
                return 0.0
            else:
                return 1.0 # if it is the same entity (same center) return maximum overlap
        
        #distance between the centers of the circles
        distance = np.linalg.norm(np.array(center2) - np.array(center1))

        if distance >= radius1 + radius2:
            return 0

        elif distance <= abs(radius1 - radius2):
            return min(np.pi * radius1**2, np.pi * radius2**2) / max(np.pi * radius1**2, np.pi * radius2**2)

        else:
            overlap = (radius1**2 * np.arccos((distance**2 + radius1**2 - radius2**2) / (2 * distance * radius1))
                    + radius2**2 * np.arccos((distance**2 + radius2**2 - radius1**2) / (2 * distance * radius2))
                    - 0.5 * np.sqrt((-distance + radius1 + radius2) * (distance + radius1 - radius2) * (distance - radius1 + radius2) * (distance + radius1 + radius2)))
            
            return float(overlap / (np.pi * min(radius1**2, radius2**2)))
        
    def calculate_cosine_similarity(self,row,centers):
        # print("center1:",centers[int(row["art1"])],"center2:",centers[int(row["art2"])])
        center_1=centers[int(row["art1"])]
        center_2=centers[int(row["art2"])]

        if center_1 == None or center_2 == None:
            return 0
        
        score=float(F.cosine_similarity(centers[int(row["art1"])], centers[int(row["art2"])], dim=0))
        # print("SCORE",score)
        return score
    
    def predict_link(self,row,centers):
        center_1=centers[int(row["art1"])]
        center_2=centers[int(row["art2"])]
        #print("CENTERS:",center_1,center_2)

        if center_1==None or center_2==None:
            return np.random.rand()
        
        head=self.entity_from_embedding(center_1)
        tail=self.entity_from_embedding(center_2)

        target_predictions = predict.predict_target(
            model=self.kge_model,
            head=head,
            tail=tail,
            triples_factory=self.triples_factory)
        
        #df["score tail_label"]
        print(target_predictions.df["score"].loc[0])
        #de momento devolver la predicción con el score más alto
        return target_predictions.df["score"].loc[0]


    def train_embeddings(self,use_trained_kgem=True,model_path=None,training_kwargs={},model_kwargs={},optimizer_kwargs={},embedding_model="transe",random_seed=20,device="gpu",split_sizes=[0.8,0.1]):
        
        if self.use_pretrained_embeddings:
            self.pretrained_embeddings=global_pretrained_embeddings
            # load dict with wikidata5m entities as keys and embeddings as values
            # with open('PRETRAINED_EMBEDDINGS/wikidata5m_embeddings.pkl', 'rb') as f:
            #     print("LOADING EMBEDDINGS DICTIONARY")
            #     self.pretrained_embeddings = pickle.load(f)
            #     #CAMBIAR A TENSORES EN VEZ NUMPY ARRAYS
            #     for k in self.pretrained_embeddings:
            #         self.pretrained_embeddings[k]=torch.tensor(self.pretrained_embeddings[k])
            #     print("EMBEDDINGS LOADED SUCCESFULLY")
        elif use_trained_kgem:
            self.kge_model=torch.load(model_path, map_location=torch.device(device))
        else:
            # TRAIN THE EMBEDDING MODEL
            training_factory, testing_factory, validation_factory = self.triples_factory.split(split_sizes)

            self.pipeline_result = pipeline(
                training=training_factory,
                validation=validation_factory,
                testing=testing_factory,
                model = embedding_model,
                model_kwargs=model_kwargs,
                random_seed=random_seed,
                device=device,
                training_kwargs=training_kwargs,
                optimizer_kwargs=optimizer_kwargs)
            
            self.kge_model=self.pipeline_result.model

    def embedding_from_entity(self,entity):
        try:
            if self.use_pretrained_embeddings:
                # devuelve embedding correspondiente en el diccionario
                return self.pretrained_embeddings[entity]
            else:
                entity_id = torch.as_tensor(self.triples_factory.entities_to_ids([entity]))
        except KeyError: # devuelve none si la entidad no está en el grafo
            return None
        except ValueError as e:
            print(entity)
            print(self.triples_factory.entities_to_ids([entity]))
            raise e

        entity_representation_modules: List['pykeen.nn.Representation'] = self.kge_model.entity_representations
        entity_embeddings: pykeen.nn.Embedding = entity_representation_modules[0]
        entity_embedding_tensor: torch.FloatTensor = entity_embeddings(indices=entity_id).detach()[0]
        
        return entity_embedding_tensor 
    
class KGECoSimRecommender(KGERecommender):
    '''Recommender that uses cosine similarity matrix between entity embeddings.'''
    def __init__(self, *args,**kwargs) -> None:
        super().__init__(*args,**kwargs)
        
    def calculate_similarity_features(self,row,embeddings):
        '''calculate similarity matrix and return dataframe with useful features (mean,max,sum,median...)'''
        
        embeddings1=embeddings[int(row["art1"])]
        embeddings2=embeddings[int(row["art2"])]
        
        if len(embeddings1)==0 or len(embeddings2)==0:
            return 0,0,0,0,0 #si no hay información de las entidades se asume que la similitud es completamente nula/ortogonal
        
        #print("EMEBDDINGS:",len(embeddings1),len(embeddings2))
        similarity_matrix = np.zeros((len(embeddings1), len(embeddings2)))

        for i in range(len(embeddings1)):
            for j in range(len(embeddings2)):
                similarity_matrix[i, j] = F.cosine_similarity(embeddings1[i],embeddings2[j],dim=0)

        media=np.mean(similarity_matrix)
        maximo=np.max(similarity_matrix)
        mediana=np.median(similarity_matrix)
        suma=np.sum(similarity_matrix)
        minimo=np.min(similarity_matrix)
        
        return media,maximo,mediana,suma,minimo
    
    def train(self,train_data,labels,news,classifier="logistic",use_trained_kgem=True,model_path=None, device="cpu", hidden_layer_sizes=(32),max_iter=200,activation="relu"):
        self.train_embeddings(use_trained_kgem=use_trained_kgem,model_path=model_path,device=device)
        
        embeddings=news[self.entity_col].apply(lambda entity_list: [self.embedding_from_entity(entity["wikidataId"]) for entity in entity_list if self.embedding_from_entity(entity["wikidataId"]) is not None]) #get entities tensor
        
        features_df=pd.DataFrame()

        features_df[["mean", "max", "median", "sum", "min"]] = train_data.apply(lambda row: pd.Series(self.calculate_similarity_features(row, embeddings)), axis=1)
        
        match classifier:
            case "logistic":
                self.classification_layer=LogisticRegression()
            case "random forest":
                self.classification_layer=RandomForestClassifier()
            case "mlp":
                self.classification_layer=MLPClassifier(hidden_layer_sizes=hidden_layer_sizes,max_iter=max_iter,activation=activation)
            case _  : 
                raise AssertionError("No Classifier layer was trained. Use classifier param to specify the model used for the classification layer.")
        
        self.classification_layer.fit(features_df,labels)

    def predict(self, test_data):
        embeddings=news[self.entity_col].apply(lambda entity_list: [self.embedding_from_entity(entity["wikidataId"]) for entity in entity_list if self.embedding_from_entity(entity["wikidataId"]) is not None]) #get entities tensor
        
        features_df=pd.DataFrame()

        features_df[["mean", "max", "median", "sum", "min"]] = test_data.apply(lambda row: pd.Series(self.calculate_similarity_features(row, embeddings)), axis=1)
        
        predictions=self.classification_layer.predict(features_df)

        return predictions

class KGEOverlapRecommender(KGERecommender):
    '''Recommender that trains with many overlapping features.'''
    def __init__(self, *args,**kwargs) -> None:
        super().__init__(*args,**kwargs)
        
    def get_center(self,embeddings,center_type="centroid",radius_type="mean distance"):
        if len(embeddings) == 0:
            return None, None

        embeddings=torch.stack(embeddings)

        match center_type:
            case "centroid": #media aritmética: coordenadas resultantes de hacer la media de todas las entidades
                center = torch.mean(embeddings, dim=0)

            case "geometric median": #media geométrica: minimiza la distancia a todas las entidades y devuelve un punto que equidista de todas
                # handle zero values and negative values
                eps = torch.finfo(embeddings.dtype).tiny
                embeddings = torch.clamp(embeddings, min=eps)  # Avoid zero values
                center = torch.prod(embeddings, axis=0) ** (1 / len(embeddings))
        
        match radius_type:
            case "max distance":
                radius = torch.max(torch.norm(embeddings - center, dim=1)).item()
            case "mean distance":
                radius = torch.mean(torch.norm(embeddings - center, dim=1)).item()
            case "median distance":
                radius = torch.median(torch.norm(embeddings - center, dim=1)).item()
            
        return center, round(radius,7)
    
    def calculate_overlap_single(self,center1,radius1,center2,radius2):

        # if there is no data in any of the clusters
        if center1==None or center2==None:
            return 0
        
        # if one cluster is a single point and the other is not
        if radius1 == 0 and radius2 > 0:
            distance = np.linalg.norm(np.array(center2) - np.array(center1))
            if distance <= radius2:
                return 1.0 # return maximum overlap if the entity falls within the cluster range
            else:
                return 0.0 # return 0 overlap if the entity does not fall within the cluster range
        elif radius2 == 0 and radius1 > 0:
            distance = np.linalg.norm(np.array(center1) - np.array(center2))
            if distance <= radius1:
                return 1.0
            else:
                return 0.0
        
        #distance between the centers of the circles
        distance = np.linalg.norm(np.array(center2) - np.array(center1))

        # if both clusters are single points
        if radius1 == radius2 == 0:
            if distance == radius1 == radius2 == 0:
                return 1.0 # overlap is maximumbecause the same entity appears in both news
            else:
                return 0.0 # two different entities

        # clusters with more than one entity
        if distance >= radius1 + radius2:
            return 0.0

        elif distance <= abs(radius1 - radius2):
            return float(min(np.pi * radius1**2, np.pi * radius2**2) / max(np.pi * radius1**2, np.pi * radius2**2))

        else:
            overlap = (radius1**2 * np.arccos((distance**2 + radius1**2 - radius2**2) / (2 * distance * radius1))
                    + radius2**2 * np.arccos((distance**2 + radius2**2 - radius1**2) / (2 * distance * radius2))
                    - 0.5 * np.sqrt((-distance + radius1 + radius2) * (distance + radius1 - radius2) * (distance - radius1 + radius2) * (distance + radius1 + radius2)))
            
            return float(overlap / (np.pi * min(radius1**2, radius2**2)))
    
    def calculate_overlap_features(self,row,embeddings):
        
        embeddings1=embeddings[int(row["art1"])]
        embeddings2=embeddings[int(row["art2"])]
        
        if len(embeddings1)==0 or len(embeddings2)==0:
            return 0,0,0,0,0.676183,0.404318,0,0,0,0,0,0,0,0 #si no hay información de las entidades se asume que no hay ningún solape
            # distancia no puede ser 0, sustituyo por la media
        
        # calcular radios y centros
        center1_median,r1_median=self.get_center(embeddings1,center_type="centroid",radius_type="median distance")
        center1_max,r1_max=self.get_center(embeddings1,center_type="centroid",radius_type="max distance")
        center1_mean,r1_mean=self.get_center(embeddings1,center_type="centroid",radius_type="mean distance")
        center1_geometric,r1_geometric=self.get_center(embeddings1,center_type="geometric median")

        center2_median,r2_median=self.get_center(embeddings2,center_type="centroid",radius_type="median distance")
        center2_max,r2_max=self.get_center(embeddings2,center_type="centroid",radius_type="max distance")
        center2_mean,r2_mean=self.get_center(embeddings2,center_type="centroid",radius_type="mean distance")
        center2_geometric,r2_geometric=self.get_center(embeddings2,center_type="geometric median")

        # calcular las ditancias entre los centros
        distance_centroids = np.linalg.norm(np.array(center2_median) - np.array(center1_median))
        distance_geometric = np.linalg.norm(np.array(center2_geometric) - np.array(center1_geometric))

        # calcular todos los solapes
        centroid_Rmean=self.calculate_overlap_single(center1_mean,r1_mean,center2_mean,r2_mean)

        centroid_Rmedian=self.calculate_overlap_single(center1_median,r1_median,center2_median,r2_median)
       
        centroid_Rmax=self.calculate_overlap_single(center1_max,r1_max,center2_max,r2_max)
       
        geometric=self.calculate_overlap_single(center1_geometric,r1_geometric,center2_geometric,r2_geometric)
        

        return centroid_Rmean,centroid_Rmedian,centroid_Rmax,geometric,distance_centroids,distance_geometric,r1_median,r1_max,r1_mean,r1_geometric,r2_median,r2_max,r2_mean,r2_geometric


class KGEFullRecommender(KGEOverlapRecommender,KGECoSimRecommender):
    '''Recommender that can use all features (cosine similarity and overlapping) for training'''
    def __init__(self, *args,**kwargs) -> None:
        super().__init__(*args,**kwargs)

    def train(self, train_data, labels, news, classifier="logistic", use_trained_kgem=True, model_path=None, device="cpu", model_params={},
              param_grid={},
              used_features=["centroid-Rmean", "centroid-Rmedian", "centroid-Rmax", "geometric", "distance_centroids", "distance_geometric","mean", "max", "median", "sum", "min"],
              task = "classification",
              regressor = "random forest"):
        
        self.train_embeddings(use_trained_kgem=use_trained_kgem,model_path=model_path,device=device)
        
        embeddings=news[self.entity_col].apply(lambda entity_list: [self.embedding_from_entity(entity["wikidataId"]) for entity in entity_list if self.embedding_from_entity(entity["wikidataId"]) is not None]) #get entities tensor
        
        features_df=pd.DataFrame()

        #first get overlap features
        features_df[["centroid-Rmean", 
                     "centroid-Rmedian", 
                     "centroid-Rmax", 
                     "geometric",
                     "distance_centroids",
                     "distance_geometric",
                     "r1_median",
                     "r1_max",
                     "r1_mean",
                     "r1_geometric",
                     "r2_median",
                     "r2_max",
                     "r2_mean",
                     "r2_geometric"
                     ]] = train_data.apply(lambda row: pd.Series(self.calculate_overlap_features(row, embeddings)), axis=1)
        
        # get the cosine similarity matrix features
        features_df[["mean", "max", "median", "sum", "min"]] = train_data.apply(lambda row: pd.Series(self.calculate_similarity_features(row, embeddings)), axis=1)

        self.used_features=used_features
        features_df=features_df[self.used_features]

        if task == "classification":

            match classifier:
                case "logistic":
                    self.classification_layer=LogisticRegression(**model_params)
                case "random forest":
                    self.classification_layer=RandomForestClassifier(**model_params)
                case "mlp":
                    self.classification_layer=MLPClassifier(**model_params)
                case _  : 
                    raise AssertionError("No Classifier layer was trained. Use classifier param to specify the model used for the classification layer.")

        elif task == "regression":
            match regressor:
                case "random forest":
                    self.classification_layer=RandomForestRegressor(**model_params)
                case _  : 
                    raise AssertionError("No Regression layer was trained. Use regressor param to specify the model used for the regression layer.")
        
        else:
            raise ValueError(f"Wrong task: '{task}'Task must be regression or classification.")

        if param_grid != {}:
            self.grid_search = GridSearchCV(estimator=self.classification_layer, param_grid=param_grid, cv=5, scoring='f1', verbose=1)
            self.grid_search.fit(features_df, labels)
            self.classification_layer = self.grid_search.best_estimator_

        else:
            self.classification_layer.fit(features_df,labels)


        return features_df # to check features
    
    def predict(self, test_data):
        embeddings=news[self.entity_col].apply(lambda entity_list: [self.embedding_from_entity(entity["wikidataId"]) for entity in entity_list if self.embedding_from_entity(entity["wikidataId"]) is not None]) #get entities tensor
        
        features_df=pd.DataFrame()

        #first get overlap features
        features_df[["centroid-Rmean", 
                     "centroid-Rmedian", 
                     "centroid-Rmax", 
                     "geometric",
                     "distance_centroids",
                     "distance_geometric",
                     "r1_median",
                     "r1_max",
                     "r1_mean",
                     "r1_geometric",
                     "r2_median",
                     "r2_max",
                     "r2_mean",
                     "r2_geometric"
                     ]] = test_data.apply(lambda row: pd.Series(self.calculate_overlap_features(row, embeddings)), axis=1)
        
        
        # get the cosine similarity matrix features
        features_df[["mean", "max", "median", "sum", "min"]] = test_data.apply(lambda row: pd.Series(self.calculate_similarity_features(row, embeddings)), axis=1)

        features_df=features_df[self.used_features]
        predictions=self.classification_layer.predict(features_df)

        return predictions

In [ ]:
def evaluate(labels, model=None, test_data=None, pred_labels=None, show_classification_report=True,task="classification"):
  
    if model:
        pred_labels = model.predict(test_data)
        model_name=model.name
    else:
        model_name=pred_labels.name

    if task=="regression":
        metrics = {
            "MAE": mean_absolute_error(labels, pred_labels),
            "MSE": mean_squared_error(labels, pred_labels),
            "RMSE": mean_squared_error(labels, pred_labels, squared=False),
            "R2": r2_score(labels, pred_labels),
            "Pearson_Corr": pearsonr(labels, pred_labels)[0],
            "Spearman_Corr": spearmanr(labels, pred_labels)[0]
        }

    if task=="classification":
        
        metrics = {
            "Accuracy": accuracy_score(labels, pred_labels),
            "Precision": precision_score(labels, pred_labels),
            "Recall": recall_score(labels, pred_labels),
            "F1": f1_score(labels, pred_labels),
            "AUC": roc_auc_score(labels, pred_labels)
        }
        print(model_name.upper())
        if show_classification_report:
            print(classification_report(labels,pred_labels))

    return metrics

# Training Example

## Classification

In [ ]:
# load triples factory
triples_factory = TriplesFactory.from_path("graphs/wikidata5m_filtered_by_entities.tsv")
# triples_factory = get_dataset("wikidata5m")


# model instance
my_kge_recommender = KGEFullRecommender("AMOR Random Forest", # a verbous name for the model
                                        triples_factory, # triples factory used to train the KGE layer
                                        use_pretrained_embeddings=True, 
                                        entity_col="text entities") # name of the column that contains the entity ids

In [ ]:
# select features for training
used_features = ["mean","max","median","sum", "min", # cosine similarity features
                 "centroid-Rmean", "centroid-Rmedian", "centroid-Rmax", "geometric","distance_centroids","distance_geometric",] # overlapping features

my_kge_recommender.train(train_pairs, # pairs df 
                         news=news, # news metadata
                         labels=train_pairs["goodR-50"],
                         classifier="random forest",
                         task = "classification")

In [ ]:
pred_labels = my_kge_recommender.predict(test_data=test_pairs)
evaluate(test_pairs["goodR-50"], my_kge_recommender, test_pairs, pred_labels)

In [ ]:
# save only the classification layer using joblib
joblib.dump(my_kge_recommender.classification_layer, filename="models/classification/random_forest.joblib")

## Regression

In [ ]:
my_kge_recommender = KGEFullRecommender("AMOR Random Forest", triples_factory, use_pretrained_embeddings=True, entity_col="text entities")

used_features = ["mean","max","median","sum", "min", # cosine similarity features
                 "centroid-Rmean", "centroid-Rmedian", "centroid-Rmax", "geometric","distance_centroids","distance_geometric",] # overlapping features

my_kge_recommender.train(train_pairs,news=news,
                         labels=train_pairs["goodR"], # use continuous score
                         regressor="random forest", 
                         task="regression") # change task to regression

In [ ]:
pred_labels = my_kge_recommender.predict(test_data=test_pairs)
evaluate(test_pairs["goodR"], my_kge_recommender, test_pairs, pred_labels, task="regression")

In [ ]:
joblib.dump(my_kge_recommender.classification_layer, filename="models/regression/random_forest.joblib")